# Seattle, WA — 50% + \$200k Building Exemption on Voted, Seattle-Contained Levies

**Center for Land Economics — LVTShift**

This model isolates the **voted property-tax levies whose taxing district is *wholly contained*
within the City of Seattle** and applies a **stacking building (improvement) exemption** to that
slice: the **first \$200,000** of every building's taxable value is fully exempt, and **50%** of
the remaining building value is exempt. A single millage is then set on the remaining
(land + un-exempted building) value so the targeted levies raise the same total — revenue-neutral
*within the targeted slice*, redistributed off buildings and onto land.

**Policy parameters (refined 2026-06-15):**

| Decision | Choice |
|---|---|
| Reform | **Building exemption: \$200k base + 50%** of remainder (toolkit `model_stacking_improvement_exemption`, `floor=200_000`, `pct=0.50`) |
| Scope | **Voted levies wholly contained within the City of Seattle** — the city excess lid-lifts, the regular lid-lift, the voted G.O. bond, and the **Seattle Metropolitan Park District**. Combined rate **2.08119 /\$1000**. |
| Excluded | **Seattle School District No. 1** (≈ coextensive with the city but *not* wholly contained), plus all state / county / port / EMS / flood / Sound Transit levies. |
| Frame | **Isolate the targeted levies** — `current_tax`/`new_tax` are the targeted slice, not the full bill |
| Existing exemptions | **Preserved** — King County post-exemption taxable values; fully-exempt parcels held out of the solver |
| Revenue target | **Derived from King County** 2025 *Codes and Levies* rate book × the parcel taxable base |

**Data:** King County GIS parcel layer + Assessor values, tax year **2025** (cached extract
`data/seattle_parcels_2025_11_20.parquet`). **Commercial building values are corrected** (Section 1b):
the raw roll carries a ~\$1,000 placeholder improvement value for many income-producing commercial
parcels (value held off-parcel on leasehold / personal-property accounts), so they are revalued by
the **cost approach** — Commercial Building file square footage × observed \$/sqft of comparable
valued buildings (taxable parcels only; prebuilt by `data/assessor/build_value_corrections.py`).
Levy rates: King County Assessor *2025 Codes and Levies* rate book (`data/levy/ratebook25.pdf`);
levy classification per the millage breakdown in `analysis/reports/seattle/millages/`.

In [ ]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_stacking_improvement_exemption,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'seattle'
STATE_FIPS = '53'              # Washington
COUNTY_FIPS = '033'            # King County
MODEL_TYPE = 'building_abatement_50pct_200k'

IMPROVEMENT_EXEMPTION_PCT = 0.50    # 50% of building value above the base is exempt
BUILDING_ABATEMENT_FLOOR = 200_000  # first $200k of each building fully exempt

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 1 — Load Parcel Data

The cached extract is the King County GIS parcel layer joined to Assessor values
(tax year 2025). We filter to parcels whose **levy jurisdiction is the City of Seattle**
(`LEVY_JURIS == 'SEATTLE'`) — the set subject to City of Seattle levies. Each parcel's
`LEVYCODE` (tax code area) determines its exact levy composition.

In [ ]:
PARCEL_PATH = DATA_DIR / 'seattle_parcels_2025_11_20.parquet'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} King County parcels from cache")
else:
    raise FileNotFoundError(
        "Expected cached King County parcel extract at "
        f"{PARCEL_PATH}. Source: King County GIS parcel layer + Assessor value "
        "extract (tax year 2025). Re-download from the KCGIS Open Data parcel "
        "service and join Assessor APPR/TAX value fields before re-running."
    )

# Filter to City of Seattle parcels (by levy jurisdiction)
gdf = gdf[gdf['LEVY_JURIS'] == 'SEATTLE'].copy()
print(f"City of Seattle parcels: {len(gdf):,}")
print("Tax code areas present:", sorted(gdf['LEVYCODE'].dropna().unique().tolist()))

In [ ]:
# Section 1b — Restore reliable commercial building values (cost approach)
# The King County parcel roll records a ~$1,000 PLACEHOLDER improvement value for many
# income-producing commercial parcels (retail, office, warehouse, restaurant, bank, ...) whose
# building value is carried off-parcel (leasehold / personal-property accounts). Left as-is they
# look like bare land and wrongly pin to the maximum increase. We restore those buildings using
# the Assessor Commercial Building file's square footage x the observed $/sqft of comparable
# Seattle buildings that DO have real assessed values (taxable parcels only — exempt
# institutional buildings are excluded). Corrections are prebuilt by
# data/assessor/build_value_corrections.py.
gdf['PIN'] = gdf['PIN'].astype(str).str.zfill(10)
gdf['TAX_IMPR'] = pd.to_numeric(gdf['TAX_IMPR'], errors='coerce').fillna(0)
gdf['imp_value_imputed'] = 0

_corr_path = DATA_DIR / 'seattle_value_corrections.parquet'
if _corr_path.exists():
    _corr = pd.read_parquet(_corr_path).set_index('PIN')['imputed_tax_impr']
    _hit = gdf['PIN'].map(_corr)
    _mask = _hit.notna() & (_hit > gdf['TAX_IMPR'])     # only raise placeholders, never lower a real value
    _added = float((_hit[_mask] - gdf.loc[_mask, 'TAX_IMPR']).sum())
    gdf.loc[_mask, 'TAX_IMPR'] = _hit[_mask].astype(float)
    gdf.loc[_mask, 'imp_value_imputed'] = 1
    print(f"Restored building values on {int(_mask.sum()):,} commercial parcels "
          f"(+${_added/1e9:.2f}B taxable improvement; cost-approach sqft x median $/sqft).")
else:
    print("WARNING: data/seattle_value_corrections.parquet not found — run "
          "data/assessor/build_value_corrections.py first. Proceeding on raw (placeholder) values.")

## Section 2 — Levy Classification & Targeted Current Tax

### Column mapping

| Concept | Column | Notes |
|---|---|---|
| Taxable land value | `TAX_LNDVAL` | Post-exemption (preserves existing exemptions) |
| Taxable improvement value | `TAX_IMPR` | Post-exemption building value |
| Appraised land / improvement | `APPRLNDVAL` / `APPR_IMPR` | Market value (reference) |
| Tax code area | `LEVYCODE` | Determines levy composition |
| Levy jurisdiction | `LEVY_JURIS` | City filter |
| Use description | `PREUSE_DESC` | Property categorization |

### Which levies are "voted *and* wholly contained within Seattle"?

From the King County Assessor **2025 Codes and Levies** rate book, cross-referenced with the
taxing-district boundary analysis in `analysis/reports/seattle/millages/`. The scope is the set
of voter-approved levies whose taxing **district boundary is exactly the City of Seattle** — so
the geography of the levy moves with the city, not a larger county/region/school district.

| Component | Rate /\$1000 | In scope? | Why |
|---|---|---|---|
| State School (Pt 1 + McCleary) | 2.24653 | ❌ | State, statewide |
| King County + its lid-lifts (Parks, Best Starts, VSHSL, Crisis Care) | 1.36182 | ❌ | Countywide |
| Port of Seattle | 0.10196 | ❌ | Countywide port district |
| Emergency Medical Services | 0.22146 | ❌ | Countywide district (EM-2 is only a sub-portion) |
| King County Flood District | 0.09757 | ❌ | Countywide |
| Sound Transit (RST1) | 0.16382 | ❌ | 3-county RTA |
| **Seattle School District No. 1** (EP&O + Capital/Bond) | 1.88116 | ❌ | Voted, but SD boundary only ≈ coextensive, **not wholly contained** |
| City of Seattle — regular (statutory) | 1.03867 | ❌ | Non-voted |
| **City of Seattle — excess lid-lifts** (Transportation, Housing, FEPP, Library, Democracy Voucher) | **1.57835** | ✅ | Voted, district = city |
| **City of Seattle — regular lid-lift** | **0.01970** | ✅ | Voter-approved lid-lift, district = city |
| **City of Seattle — voted G.O. bond** | **0.05459** | ✅ | Voted, district = city |
| **Seattle Metropolitan Park District (SMPD)** | **0.42855** | ✅ | Voted junior district, **coextensive with the city** |

**In-scope rate (uniform, all 8 Seattle TCAs):** 1.57835 + 0.01970 + 0.05459 + 0.42855 =
**2.08119 /\$1000**. Because the school levy is now out of scope, the rate no longer varies by
tax code area (the Highline-SD sliver in TCAs `0030`/`0032` is irrelevant here).

In [ ]:
# Official 2025 rate components (King County Assessor "Codes and Levies" rate book)
# IN SCOPE: VOTED property-tax levies whose taxing district is WHOLLY CONTAINED within the
# City of Seattle. Uniform across all 8 Seattle tax code areas (no school => no TCA variation).
CITY_LID_LIFT     = 1.57835   # voted excess lid-lifts (Transportation, Housing, FEPP, Library, Democracy Voucher)
CITY_REG_LID_LIFT = 0.01970   # voted regular lid-lift (folded into the regular levy)
CITY_GO_BOND      = 0.05459   # voted excess G.O. bond
SMPD_RATE         = 0.42855   # Seattle Metropolitan Park District (junior district = the city)
IN_SCOPE_RATE     = CITY_LID_LIFT + CITY_REG_LID_LIFT + CITY_GO_BOND + SMPD_RATE   # 2.08119 /$1000

# EXCLUDED: Seattle School District No. 1 (1.88116) — voted, but the SD boundary is only
# approximately coextensive with the city, not wholly contained, so it is out of scope.

# Total consolidated rate per TCA — for an all-levy reconstruction sanity check
TOTAL_RATE = {
    '0010': 9.19418, '0011': 9.19418, '0013': 9.19418,
    '0014': 9.19418, '0016': 9.19418, '0025': 9.19418,
    '0030': 11.27195, '0032': 11.27195,
}

# Clean taxable values (post-exemption)
gdf['taxable_land_value'] = pd.to_numeric(gdf['TAX_LNDVAL'], errors='coerce').fillna(0).clip(lower=0)
gdf['taxable_improvement_value'] = pd.to_numeric(gdf['TAX_IMPR'], errors='coerce').fillna(0).clip(lower=0)
gdf['taxable_total'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

# In-scope (voted, wholly-contained) rate per parcel — uniform citywide
gdf['targeted_rate'] = IN_SCOPE_RATE
gdf['total_rate'] = gdf['LEVYCODE'].map(TOTAL_RATE)
assert gdf['total_rate'].notna().all(), "Unmapped LEVYCODE — check rate table"

# current_tax = the in-scope voted-levy slice only (isolate-the-levies frame)
gdf['current_tax'] = gdf['taxable_total'] * gdf['targeted_rate'] / 1000.0

# All-levy reconstruction (validation only)
recon_all = (gdf['taxable_total'] * gdf['total_rate'] / 1000.0).sum()
print(f"Total taxable value:                 ${gdf['taxable_total'].sum():,.0f}")
print(f"All-levy reconstructed Seattle tax:  ${recon_all:,.0f}  (rate x AV, official 2025 rates)")
print(f"IN-SCOPE slice (voted, Seattle-contained @ {IN_SCOPE_RATE:.5f}/$1000): ${gdf['current_tax'].sum():,.0f}")
print(f"  City lid-lifts + reg lid-lift + GO bond: ${(gdf['taxable_total']*(CITY_LID_LIFT+CITY_REG_LID_LIFT+CITY_GO_BOND)/1000).sum():,.0f}")
print(f"  Seattle Metropolitan Park District:      ${(gdf['taxable_total']*SMPD_RATE/1000).sum():,.0f}")

## Section 3 — Property Categories & Exempt Flag

Categories are built from the Assessor `PREUSE_DESC`. Fully-exempt parcels (zero taxable
land **and** improvement — churches, public schools, government, parks) are flagged and held
out of the revenue-neutral solver. The standard `$0-improvement → Vacant Land` override is
applied to taxable parcels only; fully-exempt parcels are labeled `Exempt`.

In [ ]:
def categorize_property_type(desc: str) -> str:
    """Map King County PREUSE_DESC to a standard LVTShift property category."""
    if pd.isna(desc):
        return 'Other'
    d = str(desc).strip().lower()
    if d == '':
        return 'Other'
    if d.startswith('vacant') or 'vacant land' in d:
        return 'Vacant Land'
    if 'single family' in d:
        return 'Single Family Residential'
    if 'townhouse' in d or 'rowhouse' in d:
        return 'Townhome / Rowhouse'
    if d.startswith(('duplex', 'triplex', '4-plex', 'tri-plex')):
        return 'Small Multi-Family (2-4 units)'
    if 'condominium(residential)' in d:
        return 'Condominium'
    if 'condominium(mixed' in d or 'apartment(mixed' in d or 'mixed use' in d:
        return 'Mixed Use'
    if 'condominium(office)' in d:
        return 'Office / Commercial Condo'
    if d.startswith('apartment'):
        return 'Large Multi-Family (5+ units)'
    if any(k in d for k in ['rooming house', 'retirement', 'nursing home', 'group home',
                            'residence hall', 'dorm', 'fraternity', 'sorority', 'mobile home',
                            'houseboat', 'bed & breakfast', 'rehabilitation', 'historic prop(residence']):
        return 'Other Residential'
    if 'hotel' in d or 'motel' in d:
        return 'Hotel'
    if 'office' in d or 'medical/dental' in d or d.startswith('bank'):
        return 'Office / Commercial Condo'
    if any(k in d for k in ['retail', 'shopping ctr', 'grocery', 'conv store', 'restaurant',
                            'tavern', 'auto showroom', 'service station', 'gas station', 'car wash',
                            'mini lube', 'daycare', 'bowling', 'health club', 'movie theater',
                            'skating', 'art gallery', 'auditorium', 'golf', 'sport facility',
                            'driving range', 'vet/animal', 'mortuary', 'club']):
        return 'Retail / General Commercial'
    if any(k in d for k in ['warehouse', 'industrial', 'high tech', 'high flex', 'terminal(',
                            'air terminal', 'shell structure', 'greenhse']):
        return 'Industrial'
    if d.startswith('parking'):
        return 'Transportation - Parking'
    if any(k in d for k in ['church', 'school(', 'park, public', 'governmental', 'utility, public',
                            'utility, private', 'post office', 'hospital', 'museum', 'open space',
                            'reserve/wilderness', 'transferable dev']):
        return 'Exempt'
    if any(k in d for k in ['tideland', 'water body', 'river/creek', 'easement', 'marina']):
        return 'Other'
    return 'Other Commercial'


gdf['PROPERTY_CATEGORY'] = gdf['PREUSE_DESC'].apply(categorize_property_type)

# Fully-exempt = zero taxable land AND zero taxable improvement
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# $0-improvement (but taxable) -> Vacant Land; then exempt label takes precedence
mask_vacant = (gdf['full_exmp'] == 0) & (gdf['taxable_improvement_value'] <= 0)
gdf.loc[mask_vacant, 'PROPERTY_CATEGORY'] = 'Vacant Land'
gdf.loc[gdf['full_exmp'] == 1, 'PROPERTY_CATEGORY'] = 'Exempt'

print(f"Fully-exempt parcels held out of solver: {gdf['full_exmp'].sum():,}")
print(gdf['PROPERTY_CATEGORY'].value_counts())

## Section 4 — Current Targeted Revenue (validation)

`current_tax` already holds the targeted-levy slice. Because rates come from the official
2025 *Codes and Levies* rate book and values from the official Assessor extract, the
reconstruction *is* the King County figure by construction. We anchor plausibility against the
published 2025 county value base (\$873B; Seattle ≈ 29%).

In [ ]:
taxable_parcels = gdf[gdf['full_exmp'] == 0]
target_revenue = float(taxable_parcels['current_tax'].sum())
print(f"Targeted voted-levy revenue (Seattle, 2025): ${target_revenue:,.0f}")
print(f"  taxable parcels: {len(taxable_parcels):,}   exempt: {int(gdf['full_exmp'].sum()):,}")
print(f"  implied as share of all-levy Seattle bill: "
      f"{target_revenue / (gdf['taxable_total']*gdf['total_rate']/1000).sum()*100:.1f}%")
assert target_revenue > 0

## Section 5 — Building-Exemption Model (revenue-neutral on the targeted slice)

`model_stacking_improvement_exemption` exempts the first \$200k of each building plus 50% of the
remainder, then sets a single millage on the **remaining land + building value** so the targeted
levies raise the same total. Because the exemption is uniform across parcels, pooling the
targeted levies is equivalent to holding each levy neutral individually (they share the same
parcel base).

In [ ]:
# Exclude fully-exempt parcels from the reform AND the modeled output (held out, not added back).
n_exempt = int((gdf['full_exmp'] == 1).sum())
taxable = gdf[gdf['full_exmp'] == 0].copy()

millage_rate, new_revenue, taxable = model_stacking_improvement_exemption(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    improvement_exemption_percentage=IMPROVEMENT_EXEMPTION_PCT,   # 50% of remainder
    building_abatement_floor=BUILDING_ABATEMENT_FLOOR,           # first $200k exempt
)
gdf = taxable   # modeled parcels only — fully-exempt are excluded from export + charts

# Single uniform millage on remaining (post-exemption) value -> land and building alike
land_millage = improvement_millage = float(millage_rate)
print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"\nNew uniform millage on remaining value: {millage_rate:.5f} /$1000")
print(f"  (first ${BUILDING_ABATEMENT_FLOOR:,.0f} of each building exempt + "
      f"{IMPROVEMENT_EXEMPTION_PCT*100:.0f}% of the remainder)")
print(f"vs. combined in-scope rate today: {IN_SCOPE_RATE:.5f} /$1000 on full land+building")
print(f"Revenue neutrality: ${new_revenue:,.0f} vs target ${target_revenue:,.0f} "
      f"({(new_revenue/target_revenue-1)*100:+.4f}%)")

category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title="Seattle — 50% + $200k Building Exemption on Voted, Seattle-Contained Levies")

## Section 6 — Exploratory Chart (optional)

In [ ]:
matplotlib.use('Agg')
fig, ax = plt.subplots(figsize=(10, 6))
summary = (gdf[gdf['full_exmp'] == 0]
           .groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values())
summary.plot.barh(ax=ax, color=np.where(summary >= 0, '#c0392b', '#27ae60'))
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Seattle — Median % Change by Category\n(50% + $200k Building Exemption on Voted, Seattle-Contained Levies)')
ax.set_xlabel('Median % change in targeted-levy tax')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print("saved category_preview.png")

## Section 7 — Census Join + Standard Export

In [ ]:
# Census join — load cached King County block groups (fast FTP path), else fetch.
# King County (53033) has ~1,545 block groups; the TIGERweb per-tract fetch is very slow,
# so we cache via the Census FTP shapefile + ACS API (see data/census/).
import geopandas as _gpd
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_census_dir = DATA_DIR / 'census'
_cache_gdf = _census_dir / 'king_census_gdf.parquet'
_cache_data = _census_dir / 'king_census_data.parquet'

try:
    if _cache_gdf.exists() and _cache_data.exists():
        census_gdf = _gpd.read_parquet(_cache_gdf)
        census_data = pd.read_parquet(_cache_data)
        print(f"Loaded cached King County census: {len(census_gdf):,} block groups")
    else:
        _fips = STATE_FIPS + COUNTY_FIPS
        census_data, census_gdf = get_census_data_with_boundaries(_fips, 2022)

    gdf = match_to_census_blockgroups(gdf, census_gdf)
    if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
        gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
    if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
        gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
    print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

# Standard report — PNGs in analysis/reports/seattle/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

## Validation Summary

| Check | Result |
|---|---|
| Reform | **Building exemption: \$200k base + 50% of remainder** (uniform millage on remaining land+building value) |
| Commercial value fix | **3,581 commercial parcels** had a \$1,000 placeholder improvement restored via cost approach (+\$5.29B taxable improvement); taxable improvement base \$109.3B → **\$114.5B** |
| Revenue neutrality | \$542,363,699 modeled = \$542,363,699 target (−0.0000%) |
| In-scope slice | \$542.4M @ **2.08119 /\$1000** = City lid-lifts + reg lid-lift + voted GO bond (≈\$430.7M) + Seattle Metropolitan Park District (≈\$111.7M) |
| New millage | **2.88581 /\$1000 on remaining (post-exemption) value**, vs. the 2.08119 combined in-scope rate on full value today |
| Parcel count | 185,530 City-of-Seattle parcels (`LEVY_JURIS == SEATTLE`), tax year 2025 |
| Exempt held out of solver | 7,095 fully-exempt parcels (excluded from export + charts) |
| Census coverage | **100.0%** of parcels matched to block groups |
| PNGs generated | 7 of 7 |

**Commercial building-value correction (Section 1b).** The raw King County roll records a ~\$1,000
placeholder improvement for many income-producing commercial parcels (retail, office, warehouse,
restaurant, bank, …) whose building value is carried off-parcel (leasehold / personal-property
accounts). Of the ~22,700 placeholder parcels, only the **3,581 that have a real building**
(Commercial Building file square footage) **and are taxable** are revalued; the rest are genuinely
vacant land / surface parking and keep their land-only values. Revaluation is a cost approach:
building net sqft × the median \$/sqft (≈\$146 overall) of comparable Seattle buildings with both
real sqft and a real assessed value.

**Distribution (targeted-levy tax only).** A 50% exemption with the \$200k floor is gentler than the
75% variant — a smaller share of building value comes off, so the millage is lower (2.88581 vs.
3.24774) and every shift is smaller. Medians (taxable parcels):

- *Pay more (land-heavy):* Vacant Land **+38.7%** (genuinely all land → at the ceiling), Parking **+22.0%**, Other Commercial **+19.8%**, Retail/General Commercial **+17.9%**, Office/Commercial Condo **+12.3%**, Industrial **+8.4%**, Small Multi-Family **+5.3%**, Large Multi-Family **+4.9%**.
- *Benefit / near-flat:* Single Family Residential **−3.1%** (a modest cut — at 50% the typical home tips just below break-even), Hotel **−1.7%**, Mixed Use **−2.4%**, Townhome/Rowhouse **−22.1%**.

The increase ceiling is `2.88581 / 2.08119 − 1 = +38.7%`; only genuinely land-only parcels reach it.

**Limitations.**
1. **Commercial values are imputed, not assessor-certified** — the ~3,581 corrected parcels use a cost-approach estimate (sqft × use-median \$/sqft); far more reliable than the \$1,000 placeholder, but an estimate.
2. **Senior/disabled exemptions** — the extract has no senior flag, so the model slightly overstates these parcels' *current* in-scope tax (understating their relief).
3. **Parcel base vs. certified base** — modeled \$542.4M slice; certified real-world collection is \$600.9M (gap = business personal property + state-assessed utilities, outside the parcel file).
4. **Scope** — Seattle School District No. 1 is *excluded* (≈ coextensive with the city but not wholly contained).
5. **Frame** — `current_tax`/`new_tax` are the *targeted voted-levy slice only*, not the full property-tax bill.